# SASRec (Primary Variant)

SASRec (Kang & McAuley, ICDM 2018) is a causal-attention transformer that predicts the next item from a user's time-ordered history. This notebook
documents:
1. Model + training config.
2. Full-scale training curves.
3. Final test metrics vs every baseline and the LightGCN-HG secondary.
4. Calibrated RMSE / MAE.

All numbers are loaded from saved artefacts under `results/sasrec/`, `results/lightgcn_hg/`, and `results/baselines/`.

I chose SASRec after doing LightGCN.


In [1]:
import glob
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd

def find_repo_root(start: Path) -> Path:
    p = start.resolve()
    for cand in [p, *p.parents]:
        if (cand / "src").is_dir() and (cand / "results").is_dir():
            return cand
    raise RuntimeError(f"Couldn't find repo root from {start}")

REPO = find_repo_root(Path.cwd())
RES_BASE = REPO / "results" / "baselines"
RES_HG   = REPO / "results" / "lightgcn_hg"
RES_SAS  = REPO / "results" / "sasrec"
LOGS_SAS = REPO / "logs" / "sasrec"
print("Repo:", REPO)
for p in [RES_BASE, RES_HG, RES_SAS]:
    print(f"  {'OK' if p.exists() else '--'}  {p}")

Repo: C:\Datadrive\Hriday\Education\SJSU\2nd Sem\CMPE 256\Term Project\GitHub Repo
  OK  C:\Datadrive\Hriday\Education\SJSU\2nd Sem\CMPE 256\Term Project\GitHub Repo\results\baselines
  OK  C:\Datadrive\Hriday\Education\SJSU\2nd Sem\CMPE 256\Term Project\GitHub Repo\results\lightgcn_hg
  OK  C:\Datadrive\Hriday\Education\SJSU\2nd Sem\CMPE 256\Term Project\GitHub Repo\results\sasrec


### 1. Why SASRec - small bakeoff

Before scaling SASRec up to its primary-variant config (`dim=128`, 30 epochs), I ran a small bakeoff against two other sequential/matrix-factorisation models at a matched cheap budget - `dim=64`, 8 epochs, same 1-vs-99 evaluation protocol every other model in this repo uses. Whichever model dominated at the cheap budget got scaled up.

| Model | HR@10 | NDCG@10 | Time | Notes |
|---|---|---|---|---|
| GRU4Rec (Hidasi et al., ICLR 2016) | 0.49 | 0.29 | ~5 min | recurrent next-item predictor |
| Mult-VAE (Liang et al., WWW 2018) | 0.46 | 0.29 | ~5 min | multinomial VAE on user×item matrix |
| **SASRec (Kang & McAuley, ICDM 2018)** | **0.83** | **0.78** | **~5 min** | **chosen - scaled to dim=128 / 30 epochs** |

SASRec dominated by a wide enough margin (>2× HR@10, around 3× NDCG@10) at the same compute that scaling it up was the obvious choice. The full-scale
run lands at HR@10 = 0.8808 / NDCG@10 = 0.8392.

**Why no BERT4Rec.** Petrov & Macdonald (RecSys 2022, "A Systematic Review and Replicability Study of BERT4Rec for Sequential Recommendation") re-ran BERT4Rec under normalised eval protocols across multiple datasets and found it does not consistently beat SASRec. The lift in the original BERT4Rec paper was largely a result of its much-longer training schedule (200+ epochs vs SASRec's 30). Bidirectional masked-LM is also awkward for the "predict next future booking" framing on hotel data, where causal masking aligns naturally.

**References for this section:**
- Hidasi, Karatzoglou, Baltrunas, Tikk (2016). Session-based Recommendations with Recurrent Neural Networks. ICLR.
- Liang, Krishnan, Hoffman, Jebara (2018). Variational Autoencoders for Collaborative Filtering. WWW.
- Sun, Liu, Wu et al. (2019). BERT4Rec: Sequential Recommendation with Bidirectional Encoder Representations from Transformer. CIKM.
- Petrov & Macdonald (2022). A Systematic Review and Replicability Study of BERT4Rec for Sequential Recommendation. RecSys.
- Kang & McAuley (2018). Self-Attentive Sequential Recommendation. ICDM.

### 2. Model architecture

SASRec is a transformer *decoder* applied to a user's time-ordered
sequence of items. At each position, causal self-attention summarises
the prefix up to that point; the output at position `t` is scored
against candidate items via dot product.

Forward pass (B = batch, L = max_seqlen, D = embed dim, C = candidates):
```
    input_seq : (B, L)  int, 0=pad, item ids +1-shifted
                |
       item_emb + pos_emb    (both learned, B × L × D)
                |
           2 × TransformerEncoderLayer      causal mask + key-padding mask
                |
       h = last_position      (B, D)
                |
     score = <h, item_emb[c]>   (B, C)
```
Loss is BPR per sampled negative per position --> optimiser Adam --> cosine LR decay --> early stop on val HR@10 (patience=5).

It uses the `date` column from HotelRec (100% coverage on the 20-core), the key feature-rich angle for this variant.

## 3. Full-scale config


In [2]:
import yaml
cfg = yaml.safe_load((REPO / "configs" / "sasrec.yaml").read_text())
pd.Series({
    "embedding_dim": cfg["model"]["embedding_dim"],
    "max_seqlen":    cfg["model"]["max_seqlen"],
    "num_heads":     cfg["model"]["num_heads"],
    "num_layers":    cfg["model"]["num_layers"],
    "dropout":       cfg["model"]["dropout"],
    "epochs":        cfg["training"]["epochs"],
    "batch_size":    cfg["training"]["batch_size"],
    "lr":            cfg["training"]["lr"],
    "patience":      cfg["training"]["patience"],
    "num_negatives": cfg["negative_sampling"]["num_negatives"],
    "eval_negatives":cfg["evaluation"]["num_negatives"],
}).to_frame("value")

,value
embedding_dim,128.000
max_seqlen,100.000
num_heads,2.000
num_layers,2.000
dropout,0.200
epochs,30.000
batch_size,256.000
lr,0.001
patience,5.000
num_negatives,1.000


## 4. Training curves


In [ ]:
import matplotlib.pyplot as plt

def load_json(p):
    if not p.exists():
        return None
    with open(p) as f:
        return json.load(f)

# Try to plot the per-epoch metrics CSV first, but fall back to a one-row summary table from the saved test_metrics if the log isn't available
curves = {}
if LOGS_SAS.exists():
    for csv in sorted(LOGS_SAS.glob("metrics_*.csv")):
        curves[csv.stem.replace("metrics_", "")] = pd.read_csv(csv)

if curves:
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))
    for name, df in curves.items():
        axes[0].plot(df["epoch"], df["HR@10"], marker="o", ms=3, label=name)
        axes[1].plot(df["epoch"], df["train_loss"], marker="o", ms=3, label=name)
    axes[0].set_title("Val HR@10"); axes[0].set_xlabel("epoch"); axes[0].legend(fontsize=8)
    axes[1].set_title("Training loss (BPR)"); axes[1].set_xlabel("epoch"); axes[1].legend(fontsize=8)
    for ax in axes: ax.spines[["top","right"]].set_visible(False)
    plt.tight_layout(); plt.show()
else:
    test_meta = load_json(RES_SAS / "test_metrics_d128_L2.json")
    if test_meta is None:
        print("No SASRec results found.")
    else:
        summary = pd.DataFrame([{
            "config": f"dim={test_meta.get('embed_dim')}, L={test_meta.get('num_layers')}, heads={test_meta.get('num_heads')}",
            "max_seqlen": test_meta.get("max_seqlen"),
            "best_epoch": test_meta.get("best_epoch"),
            "best_val_HR@10": round(test_meta.get("best_val_HR@10", 0), 4),
            "test_HR@10": round(test_meta["HR@10"], 4),
            "test_NDCG@10": round(test_meta["NDCG@10"], 4),
            "train_time_s": round(test_meta.get("total_train_time_s", 0), 1),
        }]).T
        summary.columns = ["value"]
        print("SASRec full-scale run summary (per-epoch CSV not committed; logs/ is gitignored):")
        display(summary)


SASRec full-scale run summary (per-epoch CSV not committed; logs/ is gitignored):


,value
config,"dim=128, L=2, heads=2"
max_seqlen,100
best_epoch,15
best_val_HR@10,0.8807
test_HR@10,0.8808
test_NDCG@10,0.8392
train_time_s,869.4


## 5. Full-scale test metrics


In [4]:
rows = []
for p in sorted(glob.glob(str(RES_SAS / "test_metrics_d*_L*.json"))):
    m = load_json(Path(p))
    if m is None: continue
    fname = Path(p).name
    mm = re.match(r"^test_metrics_d(\d+)_L(\d+)\.json$", fname)
    if not mm: continue
    rows.append({
        "dim": int(mm.group(1)),
        "layers": int(mm.group(2)),
        **{k: round(m[k], 4) for k in ("HR@5","HR@10","HR@20","NDCG@5","NDCG@10","NDCG@20")},
        "best_epoch": m.get("best_epoch"),
        "time_s": round(m.get("total_train_time_s", 0), 1),
    })
if rows:
    display(pd.DataFrame(rows))
else:
    print("No full-scale SASRec runs yet.")

,dim,layers,HR@5,HR@10,HR@20,NDCG@5,NDCG@10,NDCG@20,best_epoch,time_s
0,128,2,0.8502,0.8808,0.9173,0.8294,0.8392,0.8484,15,869.4


## 6. SASRec vs baselines vs LightGCN-HG (secondary)


In [5]:
with open(RES_BASE / "baseline_results_20core.json") as f:
    base = json.load(f)
hg = load_json(RES_HG / "test_metrics_L1_d256_grc.json")

if rows:
    best = max(rows, key=lambda r: r["HR@10"])
    cols = ["HR@5","HR@10","HR@20","NDCG@5","NDCG@10","NDCG@20"]
    out = {
        "Popularity": {c: round(base["Popularity"][c], 4) for c in cols},
        "ItemKNN":    {c: round(base["ItemKNN"][c], 4)    for c in cols},
        "GMF":        {c: round(base["GMF"][c], 4)        for c in cols},
    }
    if hg:
        out["LightGCN-HG (secondary)"] = {c: round(hg[c], 4) for c in cols}
    out[f"SASRec (dim={best['dim']}, L={best['layers']})"] = {c: best[c] for c in cols}
    display(pd.DataFrame(out).T)

,HR@5,HR@10,HR@20,NDCG@5,NDCG@10,NDCG@20
Popularity,0.3150,0.4215,0.5538,0.2318,0.2662,0.2995
ItemKNN,0.6835,0.6870,0.7091,0.6082,0.6093,0.6150
GMF,0.5553,0.6685,0.7936,0.4498,0.4863,0.5179
LightGCN-HG (secondary),0.6460,0.7591,0.8655,0.5352,0.5718,0.5988
"SASRec (dim=128, L=2)",0.8502,0.8808,0.9173,0.8294,0.8392,0.8484


## 7. Rating prediction (calibrated RMSE / MAE)


In [6]:
rating_rows = []
rb_path = REPO / "results" / "baselines" / "rating_metrics_20core.json"
if rb_path.exists():
    rb = json.load(open(rb_path))
    for name in ("GlobalMean", "Popularity", "ItemKNN"):
        rating_rows.append((name, round(rb[name]["rmse"], 4), round(rb[name]["mae"], 4)))

for p in sorted(glob.glob(str(RES_HG / "rating_metrics_L*_d*_*.json"))):
    m = load_json(Path(p))
    if not m: continue
    mm = re.match(r"^rating_metrics_L\d+_d\d+_(\w+)\.json$", Path(p).name)
    if not mm: continue
    rating_rows.append((f"LightGCN-HG ({mm.group(1)}) calibrated",
                        round(m["rmse_calibrated"], 4),
                        round(m["mae_calibrated"], 4)))

for p in sorted(glob.glob(str(RES_SAS / "rating_metrics_d*_L*.json"))):
    m = load_json(Path(p))
    if not m: continue
    mm = re.match(r"^rating_metrics_d(\d+)_L(\d+)\.json$", Path(p).name)
    if not mm: continue
    rating_rows.append((f"SASRec (d{mm.group(1)} L{mm.group(2)}) calibrated",
                        round(m["rmse_calibrated"], 4),
                        round(m["mae_calibrated"], 4)))

pd.DataFrame(rating_rows, columns=["model", "RMSE", "MAE"])

,model,RMSE,MAE
0,GlobalMean,0.9315,0.7048
1,Popularity,0.8685,0.6749
2,ItemKNN,0.9590,0.7094
3,LightGCN-HG (grc) calibrated,0.9312,0.7025
4,LightGCN-HG (none) calibrated,0.9312,0.7025
5,SASRec (d128 L2) calibrated,0.9315,0.7047


Takeaway on RMSE: every ranking-trained model (BPR) converges to RMSE ≈ 0.93 on HotelRec because calibrating BPR scores to ratings gives a near-zero linear fit (a ≈ 0), so the predicted rating is essentially the train-set mean (around 4.08). Popularity wins at 0.8685 because item-mean already captures the 78 %-skew ratings well. This is a property of the label distribution, not a weakness of the model.

## 7. Reproducing

```bash
python -m src.train_sasrec --config configs/sasrec.yaml --kcore 20
```

Training takes ~15 min on a single RTX 5070 Ti. Best val typically lands
around epoch 15, early-stopped at ~20 (patience=5).

---

Design doc: [`../PLAN.md`](../PLAN.md).
Secondary variant: [`lightgcn_hg.ipynb`](lightgcn_hg.ipynb).
